<a href="https://colab.research.google.com/github/twillixa/SL/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import, Load and Cleaning


In [2]:
import os

# --- CONFIG: update these two lines ---
GITHUB_REPO_URL = "https://github.com/twillixa/SL.git"
CSV_PATH        = "C house price.csv"
# --------------------------------------

REPO_NAME = GITHUB_REPO_URL.split("/")[-1].replace(".git", "")

# Clone only if not already cloned (avoids errors on re-run)
if not os.path.exists(f"/content/{REPO_NAME}"):
    os.system(f"git clone {GITHUB_REPO_URL}")
    print(f"Repo cloned: {REPO_NAME}")
else:
    print(f"Repo already cloned, pulling latest changes...")
    os.system(f"cd /content/{REPO_NAME} && git pull")

# Change working directory to repo
os.chdir(f"/content/{REPO_NAME}")
print(f"Working directory: {os.getcwd()}")

# Load dataset
import pandas as pd
df = pd.read_excel("distance_matrix_1.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Repo already cloned, pulling latest changes...
Working directory: /content/SL
Dataset loaded -- 31 rows x 32 columns


,DISTANCE MATRIX,"Av. Bergières 10, 1004 Lausanne","Route de Lavaux 36, 1095 Lutry","Route de la Conversion 261, 1093 La Conversion","Av. des Boveresses 44, 1010 Lausanne","Chemin de la Colice 24, 1023 Crissier","Route de Cossonay 30, 1023 Crissier","Chemin des Lentillières 15, 1023 Crissier","Chemin de Mongevon 28, 1023 Crissier","Chemin de la Gottrause, 11, 1023 Crissier",...,"Route de Genève 25, 1180 Rolle","Rte de Crassier 9, 1262 Eysins","Av. Reverdil 6, 1260 Nyon","Rue de la Morâche 9, 1260 Nyon","Place Bel Air 8, 1260 Nyon","Rue de la Colombière 28, 1260 Nyon","Rue César Soulié 3, 1260 Nyon","Chemin de la Vuarpillière 7, 1260 Nyon","Chemin Des Chalets 9, 1279 Chavannes-de-Bogis","Rue Blavignac 17, 1227 Carouge"
0,"Av. Bergières 10, 1004 Lausanne",0.000,6.276,7.195,7.688,6.372,6.370,5.585,5.777,6.520,...,29.320,41.936,42.259,42.096,42.632,42.708,41.096,42.001,47.538,75.164
1,"Route de Lavaux 36, 1095 Lutry",6.276,0.000,1.683,10.289,13.277,13.274,12.429,12.226,12.661,...,32.935,45.551,45.874,45.711,46.248,46.324,44.711,45.616,51.153,78.246
2,"Route de la Conversion 261, 1093 La Conversion",7.195,1.683,0.000,8.605,17.877,17.874,17.029,16.826,17.261,...,39.509,52.125,52.447,52.284,52.821,52.897,51.285,52.189,57.727,84.373
3,"Av. des Boveresses 44, 1010 Lausanne",7.688,10.289,8.605,0.000,14.458,14.455,13.609,13.406,13.842,...,36.090,48.705,49.028,48.865,49.402,49.478,47.865,48.770,54.307,81.475
4,"Chemin de la Colice 24, 1023 Crissier",6.372,13.277,17.877,14.458,0.000,0.235,1.485,1.678,2.065,...,25.666,38.281,38.604,38.441,38.978,39.054,37.441,38.346,43.883,71.051


## Import of required librairies




In [3]:
pip install ortools

In [ ]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

# Extract location names and numerical distance matrix
# The first column 'DISTANCE MATRIX' contains location names, and the rest are distance values.
locations = df.iloc[:, 0].tolist()
distance_matrix_raw = df.iloc[:, 1:].values.tolist()

# Data model for OR-Tools
data = {}
data['distance_matrix'] = distance_matrix_raw
data['num_locations'] = len(data['distance_matrix'])
data['num_vehicles'] = 1
data['start_node'] = 0  # Index of the first address in the locations list
data['end_node'] = data['num_locations'] - 1  # Index of the last address in the locations list
data['stop_time_minutes_per_address'] = 3

# Create the routing index manager.
# The manager takes the number of nodes, number of vehicles, and the index of the start and end depots.
# For separate start and end depots, they must be passed as lists of node indices.
manager = pywrapcp.RoutingIndexManager(data['num_locations'],
                                       data['num_vehicles'],
                                       [data['start_node']], # Start depot as a list
                                       [data['end_node']])   # End depot as a list

# Create Routing Model.
routing = pywrapcp.RoutingModel(manager)

# Create and register a transit callback (distance evaluator).
# This function returns the distance between two nodes.
def distance_callback(from_index, to_index):
    """Returns the distance between the two nodes."""
    # Convert from routing variable Index to distance matrix NodeIndex.
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data['distance_matrix'][from_node][to_node]

# Register the distance callback with the routing model.
transit_callback_index = routing.RegisterTransitCallback(distance_callback)

# Define the cost of each arc (segment of the route).
# The cost is set to the distance returned by the transit callback.
# This implicitly sets the objective to minimize the total arc cost.
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# No explicit SetStart/SetEnd needed here, as the start and end depots
# are already specified in the RoutingIndexManager.

# Add penalties for unvisited nodes to ensure all nodes (except the start depot) are visited.
# This makes visiting them mandatory. A large penalty forces the solver to include them.
penalty = 1000000000  # A very large penalty for skipping a node.
for node_idx in range(data['num_locations']):
    # Exclude only the start_node (depot) from the disjunction, as it's handled by manager.
    # The end_node must also be visited as part of the route, so it's included here.
    if node_idx == data['start_node']:
        continue
    # Add a disjunction for each intermediate node and the end_node to ensure it is visited.
    routing.AddDisjunction([manager.NodeToIndex(node_idx)], penalty)

# Set search parameters to guide the solver.
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
# Use a good first solution strategy. PATH_CHEAPEST_ARC is a common and effective heuristic.
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
# Use a metaheuristic to improve the solution locally. GUIDED_LOCAL_SEARCH is generally effective for TSP.
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
# Set a time limit for the search to prevent it from running indefinitely on large instances.
search_parameters.time_limit.FromSeconds(60) # Time limit increased to 60 seconds

# Solve the problem.
print("Solving OTSP model...")
solution = routing.SolveWithParameters(search_parameters)

# Function to print the solution and calculate total time.
def print_solution(data, manager, routing, solution):
    """Prints the solution (route and total distance) and estimates total time."""
    print(f'\n--- Solution Found ---\n')
    # The objective value will report the total arc cost, which is the total distance.
    # The previous 0.00 was an artifact of misconfiguration, the solver now minimizes this implicitly.
    print(f'Objective (Total Distance reported by solver): {solution.ObjectiveValue():.2f} units')

    total_distance_calculated = 0
    route_nodes_names = []
    route_indices = []

    vehicle_id = 0
    index = routing.Start(vehicle_id)
    plan_output = 'Route for vehicle 0:\n'

    # Iterate through the route to reconstruct the path and calculate total distance.
    while not routing.IsEnd(index):
        node_index = manager.IndexToNode(index)
        route_nodes_names.append(locations[node_index])
        route_indices.append(node_index)
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        # Accumulate distance for each segment of the route.
        total_distance_calculated += data['distance_matrix'][manager.NodeToIndex(previous_index)][manager.NodeToIndex(index)]

    # Add the last node to the route.
    last_node_index = manager.IndexToNode(index)
    route_nodes_names.append(locations[last_node_index])
    route_indices.append(last_node_index)

    plan_output += ' -> '.join(route_nodes_names)
    print(plan_output)
    print(f'Route distance calculated (sum of segments, for verification): {total_distance_calculated:.2f} units')

    # Calculate total time including stops.
    # Assumption: distance is in km, and average speed is 30 km/h (0.5 km/minute).
    # Therefore, travel time in minutes = distance / 0.5 = distance * 2.
    travel_time_minutes = 0
    for i in range(len(route_indices) - 1):
        from_node_idx = route_indices[i]
        to_node_idx = route_indices[i+1]
        travel_distance_segment = data['distance_matrix'][from_node_idx][to_node_idx]
        travel_time_minutes += travel_distance_segment * 2

    # "for each address a stop of 3 minutes is considered"
    # This implies a stop at every unique address visited on the route.
    num_unique_stops = len(set(route_indices))
    total_stop_time_minutes = num_unique_stops * data['stop_time_minutes_per_address']
    total_route_time_minutes = travel_time_minutes + total_stop_time_minutes

    print(f'\n--- Time Estimation ---\n')
    print(f'Total estimated travel time (assuming 30 km/h): {travel_time_minutes:.2f} minutes')
    print(f'Total stop time ({data["stop_time_minutes_per_address"]} min/stop for {num_unique_stops} unique stops): {total_stop_time_minutes} minutes')
    print(f'Total estimated route time (travel + stops): {total_route_time_minutes:.2f} minutes')


# Check if a solution was found and print it.
if solution:
    print_solution(data, manager, routing, solution)
else:
    print('No solution found !')

In [2]:
import pandas as pd
df = pd.read_excel("distance_matrix_2.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'distance_matrix_2.xlsx'